# Анализ кредитного риска и предсказание дефолта

**Автор:** Михаил Богданов  
**Дата:** Ноябрь 2026  
**Датасет:** [Credit Risk Dataset — Kaggle](https://www.kaggle.com/datasets/laotse/credit-risk-dataset)

---

## Цель проекта

Построить модель машинного обучения, которая предсказывает вероятность дефолта по кредиту на основе характеристик заёмщика и параметров кредита. Такая модель применяется в банках и финансовых организациях для оценки кредитного риска перед выдачей кредита.

## Структура проекта

1. Импорт библиотек и загрузка данных  
2. Первичный осмотр данных  
3. Очистка данных (пропуски, выбросы)  
4. Разведочный анализ (EDA)  
5. Подготовка данных для модели  
6. Обучение модели логистической регрессии  
7. Оценка качества модели  
8. Выводы

## 1. Импорт библиотек

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Настройка отображения графиков
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 2. Загрузка и первичный осмотр данных

In [ ]:
df = pd.read_csv('credit_risk_dataset.csv')
df.head()

**Описание колонок:**

| Колонка | Описание |
|---|---|
| `person_age` | Возраст заёмщика |
| `person_income` | Годовой доход |
| `person_home_ownership` | Тип жилья (RENT / OWN / MORTGAGE) |
| `person_emp_length` | Стаж работы (лет) |
| `loan_intent` | Цель кредита |
| `loan_grade` | Рейтинг кредита (A — лучший, G — худший) |
| `loan_amnt` | Сумма кредита |
| `loan_int_rate` | Процентная ставка |
| `loan_status` | **Целевая переменная:** 1 = дефолт, 0 = платит |
| `loan_percent_income` | Платёж как % от дохода |
| `cb_person_default_on_file` | Был ли дефолт в прошлом (Y / N) |
| `cb_person_cred_hist_length` | Длина кредитной истории (лет) |

In [ ]:
print(f'Размер датасета: {df.shape[0]} строк, {df.shape[1]} колонок')
print()
print('Типы данных:')
print(df.dtypes)

## 3. Очистка данных

### 3.1 Пропущенные значения

In [ ]:
print('Пропуски по колонкам:')
print(df.isna().sum())

Пропуски обнаружены в двух колонках:
- `loan_int_rate` — 3116 пропусков (процентная ставка)
- `person_emp_length` — 895 пропусков (стаж работы)

Для заполнения пропусков используем **медиану** — она устойчива к выбросам в отличие от среднего значения.

In [ ]:
print(f"Медиана loan_int_rate: {df['loan_int_rate'].median()}")
print(f"Медиана person_emp_length: {df['person_emp_length'].median()}")

df['loan_int_rate'] = df['loan_int_rate'].fillna(df['loan_int_rate'].median())
df['person_emp_length'] = df['person_emp_length'].fillna(df['person_emp_length'].median())

print()
print('Пропуски после заполнения:')
print(df.isna().sum())

### 3.2 Выбросы по возрасту

In [ ]:
print('Строки с возрастом > 80 лет:')
print(df[df['person_age'] > 80][['person_age', 'person_income', 'loan_amnt', 'loan_status']])

Обнаружено 7 записей с аномальным возрастом (123, 144 года) — это явные ошибки ввода данных. Строки с возрастом 84 и 94 года оставляем как реалистичные, убираем только записи со значениями 100 и выше.

In [ ]:
df = df[df['person_age'] < 100]

print(f'Максимальный возраст после очистки: {df["person_age"].max()}')
print(f'Итоговый размер датасета: {df.shape[0]} строк')

## 4. Разведочный анализ данных (EDA)

### 4.1 Распределение целевой переменной

In [ ]:
counts = df['loan_status'].value_counts()
print('Распределение по статусу кредита:')
print(f'  Платит (0): {counts[0]} ({counts[0]/len(df)*100:.1f}%)')
print(f'  Дефолт (1): {counts[1]} ({counts[1]/len(df)*100:.1f}%)')

Датасет несбалансирован: 78% заёмщиков платят исправно, 22% допускают дефолт. Дисбаланс умеренный — работать можно, но при оценке модели важно смотреть не только на accuracy, но и на recall по классу 1.

### 4.2 Распределение возраста по статусу кредита

In [ ]:
df[df['loan_status'] == 0]['person_age'].hist(alpha=0.5, label='Платит', bins=30)
df[df['loan_status'] == 1]['person_age'].hist(alpha=0.5, label='Дефолт', bins=30)
plt.legend()
plt.xlabel('Возраст')
plt.ylabel('Количество заёмщиков')
plt.title('Распределение возраста по статусу кредита')
plt.tight_layout()
plt.show()

**Вывод:** Оба распределения смещены влево — большинство заёмщиков молодые (20–35 лет). Дефолты концентрируются в той же возрастной группе, что логично: молодые заёмщики имеют меньший доход и менее стабильную занятость.

### 4.3 Распределение процентной ставки по статусу кредита

In [ ]:
df[df['loan_status'] == 0]['loan_int_rate'].hist(alpha=0.5, label='Платит', bins=30)
df[df['loan_status'] == 1]['loan_int_rate'].hist(alpha=0.5, label='Дефолт', bins=30)
plt.legend()
plt.xlabel('Процентная ставка (%)')
plt.ylabel('Количество заёмщиков')
plt.title('Распределение процентной ставки по статусу кредита')
plt.tight_layout()
plt.show()

**Вывод:** Чёткая закономерность — заёмщики с дефолтом получают кредиты под более высокие ставки (пик около 13–17%), тогда как исправные плательщики сосредоточены в диапазоне 10–12%. Банки заранее закладывают риск в ставку, и он действительно реализуется.

### 4.4 Доля дефолтов по цели кредита

In [ ]:
intent_default = df.groupby('loan_intent')['loan_status'].mean()
print('Доля дефолтов по цели кредита:')
print(intent_default.sort_values(ascending=False).round(3))

In [ ]:
intent_default.sort_values(ascending=False).plot(kind='bar', color='steelblue')
plt.title('Доля дефолтов по цели кредита')
plt.ylabel('Доля дефолтов')
plt.xlabel('Цель кредита')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Вывод:** Наибольшая доля дефолтов у кредитов на рефинансирование долгов (DEBTCONSOLIDATION — 28.6%) и медицинские расходы (MEDICAL — 26.7%). Это объяснимо: люди, берущие кредит для покрытия старых долгов, уже находятся в финансово уязвимом положении. Наименьший риск — венчурные кредиты (VENTURE — 14.8%), вероятно, из-за более тщательного отбора заёмщиков.

## 5. Подготовка данных для модели

### 5.1 Кодирование категориальных переменных

Модели машинного обучения работают только с числовыми данными. Текстовые категории преобразуем методом one-hot encoding — каждое уникальное значение становится отдельной бинарной колонкой.

In [ ]:
print('Категориальные колонки до кодирования:')
print(df.select_dtypes(include='object').columns.tolist())

df = pd.get_dummies(df, columns=['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file'])

print(f'\nРазмер датасета после кодирования: {df.shape[0]} строк, {df.shape[1]} колонок')

### 5.2 Разделение на признаки и целевую переменную

In [ ]:
X = df.drop('loan_status', axis=1)  # признаки
y = df['loan_status']               # целевая переменная

print(f'Признаки (X): {X.shape}')
print(f'Целевая переменная (y): {y.shape}')

### 5.3 Разделение на обучающую и тестовую выборки

Модель обучается на 80% данных, а оценивается на 20%, которые она не видела во время обучения. Это позволяет честно оценить способность модели к обобщению.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Обучающая выборка: {X_train.shape[0]} строк')
print(f'Тестовая выборка:  {X_test.shape[0]} строк')

## 6. Обучение модели

Используем **логистическую регрессию** — базовую модель для задач бинарной классификации. Несмотря на простоту, она хорошо интерпретируема и даёт понятные вероятностные предсказания.

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print('Модель обучена.')

## 7. Оценка качества модели

In [ ]:
y_pred = model.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print()
print('Детальный отчёт:')
print(classification_report(y_test, y_pred, target_names=['Платит (0)', 'Дефолт (1)']))

**Интерпретация результатов:**

| Метрика | Платит (0) | Дефолт (1) |
|---|---|---|
| Precision | 0.86 | 0.78 |
| Recall | 0.97 | 0.43 |
| F1-score | 0.91 | 0.55 |

- **Accuracy 85%** — модель правильно классифицирует 85% заёмщиков.
- **Recall по дефолту = 0.43** — модель находит только 43% реальных дефолтов. Это главная проблема: 57% потенциальных дефолтников модель пропускает.
- **Precision по дефолту = 0.78** — когда модель предсказывает дефолт, она права в 78% случаев.

Для банковского применения низкий recall по дефолту критичен — пропущенный дефолт обходится дороже, чем ложная тревога. Это направление для дальнейшего улучшения модели.

## 8. Выводы

В ходе проекта была построена модель предсказания кредитного дефолта на основе логистической регрессии.

**Ключевые находки из EDA:**
- Кредиты с высокой процентной ставкой (>13%) дефолтируют значительно чаще — ставка является одним из сильнейших предикторов риска.
- Кредиты на рефинансирование долгов (DEBTCONSOLIDATION) дефолтируют в 28.6% случаев — наибольший риск среди всех категорий.
- Молодые заёмщики (20–30 лет) составляют основную массу как плательщиков, так и дефолтников.

**Результаты модели:**
- Accuracy: 85%
- Модель хорошо распознаёт надёжных заёмщиков (recall 0.97), но пропускает половину реальных дефолтов (recall 0.43).

**Направления для улучшения:**
- Попробовать Random Forest или Gradient Boosting — как правило дают лучший recall по дефолту.
- Нормализовать числовые признаки (StandardScaler) — может улучшить сходимость логрегрессии.
- Применить балансировку классов (class_weight='balanced') для улучшения recall по дефолту.